# 13 — RLHF: PPO Fine-Tuning

With a trained reward model, stage 2 of RLHF fine-tunes the LLM policy using PPO. Three design choices distinguish RLHF-PPO from standard PPO: the KL penalty, the reference model, and instability management.

## The RLHF-PPO objective

$$J(\pi_{\theta}) = \mathbb{E}\left[r(x,y) - \beta \cdot KL(\pi_\theta(\cdot|x)\ \|\  \pi_{SFT}(\cdot|x))\right]$$

The KL term penalises deviating from the SFT reference model. β controls the tradeoff:
- β=0: pure reward maximisation → reward hacking, degraded language quality
- β→∞: no improvement from SFT
- β≈0.01–0.1: the practical range

The **reference model** (frozen SFT) computes log π_SFT(y|x) token-by-token for each generated sequence. The KL is summed over generated tokens.

**Instability issues**: RLHF-PPO is more unstable than standard PPO. Common failures: reward hacking divergence (KL grows rapidly), KL collapse (penalty too large, policy barely moves), gradient spikes from large LLM parameters. Mitigations: gradient clipping, small LR, KL threshold early stopping.

In [ ]:
# RLHF-PPO fine-tuning loop with a small transformer

import torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class TinyLM(nn.Module):
    def __init__(self, vocab_size=100, embed_dim=64, n_heads=4, n_layers=2, max_len=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        enc_layer = nn.TransformerEncoderLayer(embed_dim, n_heads, dim_feedforward=128,
                                               batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, n_layers)
        self.head = nn.Linear(embed_dim, vocab_size)
        self.vocab_size = vocab_size; self.max_len = max_len

    def forward(self, tokens):
        B, T = tokens.shape
        pos = torch.arange(T, device=tokens.device).unsqueeze(0)
        x = self.embed(tokens) + self.pos_embed(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=tokens.device)
        x = self.transformer(x, mask=mask, is_causal=True)
        return self.head(x)

    def generate(self, prompt, max_new_tokens=8):
        tokens = prompt.clone()
        for _ in range(max_new_tokens):
            if tokens.shape[1] >= self.max_len: break
            logits = self(tokens)
            next_tok = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            tokens = torch.cat([tokens, next_tok], dim=1)
        return tokens

    def log_probs(self, tokens):
        logits = self(tokens[:, :-1])
        lp = F.log_softmax(logits, dim=-1)
        return lp.gather(2, tokens[:, 1:].unsqueeze(-1)).squeeze(-1)

class SyntheticRM:
    def __call__(self, tokens):
        return (tokens.float().mean(dim=-1) / 50 - 1.0)

VOCAB_SIZE = 100; PROMPT_LEN = 4; GEN_LEN = 8; BETA = 0.05

sft_model = TinyLM(VOCAB_SIZE).to(device)
policy = TinyLM(VOCAB_SIZE).to(device)
policy.load_state_dict(sft_model.state_dict())
for p in sft_model.parameters(): p.requires_grad_(False)
optimizer = optim.Adam(policy.parameters(), lr=1e-4)
rm = SyntheticRM()

rewards_history, kl_history = [], []

for step in range(300):
    prompts = torch.randint(0, VOCAB_SIZE, (16, PROMPT_LEN), device=device)
    with torch.no_grad():
        full_tokens = policy.generate(prompts, max_new_tokens=GEN_LEN)
        generated = full_tokens[:, PROMPT_LEN:]
        sft_lp = sft_model.log_probs(full_tokens)[:, PROMPT_LEN-1:]

    reward = rm(generated)
    policy_lp = policy.log_probs(full_tokens)[:, PROMPT_LEN-1:]
    kl_per_seq = (policy_lp - sft_lp.detach()).sum(dim=-1)
    total_reward = reward - BETA * kl_per_seq
    baseline = total_reward.mean().detach()
    pg_loss = -(policy_lp.sum(dim=-1) * (total_reward - baseline).detach()).mean()
    optimizer.zero_grad(); pg_loss.backward(); optimizer.step()

    rewards_history.append(reward.mean().item())
    kl_history.append(kl_per_seq.mean().item())
    if (step+1) % 50 == 0:
        print(f"  Step {step+1:4d} | Reward: {np.mean(rewards_history[-50:]):+.4f} | KL: {np.mean(kl_history[-50:]):.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
w = 20
ax1.plot(rewards_history, alpha=0.3, color='steelblue')
ax1.plot(np.convolve(rewards_history, np.ones(w)/w, 'valid'), color='steelblue', label='Reward')
ax1.set_xlabel('Step'); ax1.set_title('RLHF-PPO: Reward', fontweight='bold'); ax1.legend()
ax2.plot(kl_history, alpha=0.3, color='darkorange')
ax2.plot(np.convolve(kl_history, np.ones(w)/w, 'valid'), color='darkorange', label='KL')
ax2.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='KL limit')
ax2.set_xlabel('Step'); ax2.set_title('KL Divergence from SFT', fontweight='bold'); ax2.legend()
plt.tight_layout(); plt.savefig('/tmp/rlhf_ppo.png', dpi=100, bbox_inches='tight'); plt.show()


## Production RLHF at scale

Full-scale RLHF differs in:
- **Model size**: 7B–70B+ parameter LLM; reward model same architecture with scalar head
- **Rejection sampling fine-tuning**: an alternative to PPO — sample many responses, keep top-K by reward, fine-tune on those. Simpler, more stable, sometimes competitive
- **Constitutional AI**: self-critique + revision replaces or augments human preference labels
- **Libraries**: `trl` (HuggingFace), `OpenRLHF`, `DeepSpeed-Chat` implement production RLHF pipelines